In [1]:
import numpy as np
import pandas as pd
from scipy.stats import linregress, pearsonr

from armored.models import *
from armored.preprocessing import *

import itertools

from tqdm import tqdm

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt


params = {
    'figure.figsize': (7, 5),       # Figure size in inches
    'figure.dpi': 300,               # Resolution in dots per inch
    'axes.labelsize': 7,            # Font size of the axes labels
    'axes.titlesize': 7,            # Font size of the subplot titles
    'axes.titlepad': 10,             # Spacing between the subplot title and the plot
    'axes.linewidth': .5,           # Width of the axis lines
    'axes.grid': False,              # Display grid lines
    'axes.grid.axis': 'both',        # Display grid lines for both major and minor ticks
    'grid.alpha': 0.5,               # Transparency of the grid lines
    'grid.linewidth': 0.5,           # Width of the grid lines
    'xtick.labelsize': 7,           # Font size of the x-axis tick labels
    'ytick.labelsize': 7,           # Font size of the y-axis tick labels
    'xtick.major.size': 3,           # Length of the major x-axis ticks in points
    'xtick.major.width': .5,        # Width of the major x-axis ticks
    'ytick.major.size': 3,           # Length of the major y-axis ticks in points
    'ytick.major.width': .5,        # Width of the major y-axis ticks
    'xtick.minor.size': 2,           # Length of the minor x-axis ticks in points
    'xtick.minor.width': .5,        # Width of the minor x-axis ticks
    'ytick.minor.size': 2,           # Length of the minor y-axis ticks in points
    'ytick.minor.width': .5,        # Width of the minor y-axis ticks
    'xtick.direction': 'in',         # Direction of the x-axis ticks ('in', 'out', or 'inout')
    'ytick.direction': 'in',         # Direction of the y-axis ticks ('in', 'out', or 'inout')
    'legend.fontsize': 7,           # Font size of the legend
    'legend.frameon': True,          # Whether to draw a frame around the legend
    'legend.facecolor': 'inherit',   # Background color of the legend
    'legend.edgecolor': '0.8',       # Edge color of the legend
    'legend.framealpha': 0.8,        # Transparency of the legend frame
    'legend.loc': 'best',            # Location of the legend ('best', 'upper right', 'upper left', 'lower left', etc.)
    'legend.title_fontsize': 7,     # Font size of the legend title
    'svg.fonttype': 'none'           # Output font type for PDF files
}

plt.rcParams.update(params)

In [2]:
species = ['AC', 'BA', 'BH', 'BL', 'BU', 'CA', 'CC', 'CH',
           'DF', 'EL', 'ER', 'FP', 'PC', 'PJ', 'RI']
controls = ['AcGum', 'ArGal', 'Inulin', 'Pectin', 'Starch', 'Xylan']

In [3]:
# data with initial and end point measurements
df_exp0 = pd.read_csv("data/exp0/exp0_metabolites.csv")

In [4]:
# mono-species in single fiber
def exp_type(exp_name, sp_cond=1, f_cond=1):
    
    components = exp_name.split("-")
    
    # number of species in exp 
    n_species = sum(np.isin(species, components))
    
    # number of fibers in exp
    n_fibers = sum(np.isin(controls, components))
    
    if n_species == sp_cond and n_fibers == f_cond:
        return True
    else:
        return False

In [5]:
# conditions with single species, single fiber
sp_inds = [exp_type(exp_name, 1, 1) for exp_name in df_exp0.Experiments.values]
mono_df = df_exp0.iloc[sp_inds].copy()

In [6]:
# count number of times species grew more than 50X
folds = []
for exp_name, exp_df in mono_df.groupby("Experiments"):
    
    # name of species in condition
    sps = np.array(species)[np.isin(species, exp_name.split('-'))]
    sps = np.array([s+'abs' for s in sps])
    
    # fold change
    ods = np.sum(exp_df[sps].values, 1)
    fold = ods[-1] / ods[0]
    folds.append(fold)

In [7]:
sum(np.array(folds)>50)

37

In [8]:
# conditions with LOO species, single fiber
loo_inds = [exp_type(exp_name, 14, 1) for exp_name in df_exp0.Experiments.values]
loo_df = df_exp0.iloc[loo_inds].copy()

In [9]:
# count number of times species grew more than 50X
folds = []
f_ods = []
for exp_name, exp_df in loo_df.groupby("Experiments"):
    
    # name of species in condition
    sps = np.array(species)[np.isin(species, exp_name.split('-'))]
    sps = np.array([s+'abs' for s in sps])
    
    # fold change
    ods = np.sum(exp_df[sps].values, 1)
    fold = ods[-1] / ods[0]
    folds.append(fold)
    f_ods.append(ods[-1])

In [10]:
sum(np.array(folds)>50)

89

In [19]:
# In what fraction of communities (mono in single fibers) did FP, RI, and ER grow? 
fp_grow = []
ri_grow = []
er_grow = []

for exp_name, exp_df in mono_df.groupby("Experiments"):
    
    # fold change
    if 'FP' in exp_name:
        fp_ods = exp_df["FPabs"].values
        fp_grow.append(fp_ods[-1]/fp_ods[0])
        
    # fold change
    if 'RI' in exp_name:
        ri_ods = exp_df["RIabs"].values
        ri_grow.append(ri_ods[-1]/ri_ods[0])
        
    # fold change
    if 'ER' in exp_name:
        er_ods = exp_df["ERabs"].values
        er_grow.append(er_ods[-1]/er_ods[0])

In [20]:
len(er_grow)

6

In [21]:
sum(np.array(er_grow)>50)

2

In [22]:
sum(np.array(fp_grow)>50)

0

In [14]:
sum(np.array(ri_grow)>50)

1

In [24]:
# In what fraction of communities (LOO in single fibers) did FP, RI, and ER grow? 
fp_grow = []
ri_grow = []
er_grow = []

for exp_name, exp_df in loo_df.groupby("Experiments"):
    
    # fold change
    if 'FP' in exp_name:
        fp_ods = exp_df["FPabs"].values
        fp_grow.append(fp_ods[-1]/fp_ods[0])
        
    # fold change
    if 'RI' in exp_name:
        ri_ods = exp_df["RIabs"].values
        ri_grow.append(ri_ods[-1]/ri_ods[0])
        
    # fold change
    if 'ER' in exp_name:
        er_ods = exp_df["ERabs"].values
        er_grow.append(er_ods[-1]/er_ods[0])

In [25]:
len(er_grow)

84

In [26]:
sum(np.array(er_grow)>50)

59

In [27]:
sum(np.array(fp_grow)>50)

53

In [28]:
sum(np.array(ri_grow)>50)

8